# Practice Lab #03 — Multi-Class Vietnamese Currency Classification & Analysis
**Dataset:** 6 Active Vietnamese Polymer Banknotes — **Background-Removed / Bounding-Box Cropped** (`10k`, `20k`, `50k`, `100k`, `200k`, `500k VND` — 1,308 RGB banknote crops)  
**Workflow:** Clone source code from GitHub (`huylasiunhan/xlav`) + Mount Google Drive (`MyDrive/XLAV_Lab3`) for cropped dataset & output checkpoints.

This notebook runs the complete Lab #03 training & experimental pipeline and saves checkpoints & report figures into `MyDrive/XLAV_Lab3/outputs/`:
1. **Environment Setup:** Clone/pull `xlav` from GitHub, mount Drive, unzip `vietnamese_currency_6classes_cropped.zip` to local Colab SSD (`/content/data`).
2. **Visualizations:** Cropped dataset sample grid (`01_dataset_samples.png`) & System Architecture Diagram (`02_system_architecture.png`).
3. **Model 1 — `MultiClassImageMLP` (Lab #02 Upgrade):** Direct upgrade from Binary MLP -> 6-Class MLP (`best_mlp.pt`).
4. **Model 2 — `CurrencyCNN`:** Lightweight from-scratch 2D CNN (`128×128`, ~0.95 MB, `best_cnn.pt`).
5. **Model 3 — `CurrencyMobileNetV2` (Transfer Learning, Flagship):** Pretrained ImageNet MobileNetV2 (`224×224`, `best_mobilenet.pt`) for rock-solid live Webcam & Video recognition!
6. **Comparative Analysis #1:** Binary (Lab #02) vs. Multi-Class (Lab #03) + 3-Architecture comparison (`MLP` vs `CNN` vs `MobileNetV2`).
7. **Comparative Analysis #2:** Impact of Module 1 OpenCV Filters (Color, Geometric, Smoothing) on Module 2 Multi-Class AI pipeline.


## 1. Environment Setup (GitHub Clone + Google Drive Cropped Dataset)


In [ ]:
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/huylasiunhan/xlav.git"
REPO_BRANCH = "main"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # 1. Clone (or pull latest) source code from GitHub
    REPO_DIR = Path("/content/xlav")
    if not REPO_DIR.is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

    SOURCE_DIR = REPO_DIR / "lab3" / "source"

    # 2. Load background-removed cropped banknote zip from Drive and save outputs back to Drive
    DRIVE_ROOT = Path("/content/drive/MyDrive/XLAV_Lab3")
    ZIP_PATH = DRIVE_ROOT / "vietnamese_currency_6classes_cropped.zip"
    DATA_DIR = Path("/content/data/vietnamese_currency_6classes_cropped")
    OUTPUT_DIR = DRIVE_ROOT / "outputs"

    if not (DATA_DIR / "10000_VND").is_dir():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_DIR)
        print(f"Extracted cropped dataset to {DATA_DIR}")
    else:
        print(f"Cropped dataset already extracted at {DATA_DIR}")
else:
    SOURCE_DIR = Path.cwd()
    DATA_DIR = SOURCE_DIR.parent / "data" / "vietnamese_currency_6classes_cropped"
    OUTPUT_DIR = SOURCE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

cpu_count = os.cpu_count() or 2
OPTIMAL_WORKERS = min(4, max(2, cpu_count // 2)) if (IN_COLAB and torch.cuda.is_available()) else 0

gpu_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else ("Apple Silicon MPS" if torch.backends.mps.is_available() else "CPU")
)
print(f"IN_COLAB        : {IN_COLAB}")
print(f"Compute Device  : {gpu_name}")
print(f"CPU vCores      : {cpu_count} -> num_workers = {OPTIMAL_WORKERS}")
print(f"SOURCE_DIR      : {SOURCE_DIR}")
print(f"DATA_DIR        : {DATA_DIR}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

## 2. Cropped Dataset Verification, Sample Grid & System Block Architecture


In [ ]:
from PIL import Image

from src.config import CLASS_DISPLAY_NAMES, DEFAULT_CLASS_NAMES, Lab3Config, get_device, seed_everything
from src.dataset import collect_image_paths_and_labels, create_dataloaders, split_dataset
from src.img_processing import ImageProcessingParams
from src.inference import CurrencyInferenceEngine
from src.metrics import evaluate_multiclass_metrics
from src.model import MultiClassImageMLP, build_model
from src.trainer import MultiClassTrainer, TrainingHistory
from src.transform import build_augmentation_pipeline

seed_everything(42)
paths, labels, class_to_idx = collect_image_paths_and_labels(str(DATA_DIR), DEFAULT_CLASS_NAMES)
(x_tr, y_tr), (x_va, y_va), (x_te, y_te) = split_dataset(paths, labels, 0.70, 0.15, 0.15, seed=42)

print(f"Total cropped banknote images: {len(paths)} across {len(DEFAULT_CLASS_NAMES)} classes")
print(f"Split sizes : Train={len(y_tr)}, Val={len(y_va)}, Test={len(y_te)}")
for cname, idx in class_to_idx.items():
    print(
        f"  {CLASS_DISPLAY_NAMES[cname]:<12} | "
        f"Train={y_tr.count(idx):>3d} | Val={y_va.count(idx):>2d} | Test={y_te.count(idx):>2d}"
    )

In [ ]:
# Plot 01: Original Cropped vs. Augmented sample grid for all 6 VND denominations
aug_pipe = build_augmentation_pipeline()
fig, axes = plt.subplots(len(DEFAULT_CLASS_NAMES), 3, figsize=(10, 12))
col_titles = ["Cropped Banknote (No BG)", "Augmented View #1", "Augmented View #2"]

for row_idx, cname in enumerate(DEFAULT_CLASS_NAMES):
    cls_id = class_to_idx[cname]
    first_idx = labels.index(cls_id)
    with Image.open(paths[first_idx]) as im:
        raw_rgb = np.array(im.convert("RGB"))

    views = [raw_rgb, aug_pipe(raw_rgb.copy()), aug_pipe(raw_rgb.copy())]
    for col_idx, arr in enumerate(views):
        ax = axes[row_idx, col_idx]
        ax.imshow(arr)
        ax.set_xticks([])
        ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=11, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(CLASS_DISPLAY_NAMES[cname], fontsize=10, fontweight="bold")

fig.suptitle("Cropped Vietnamese Polymer Currency Dataset (6 Classes): Clean vs. Augmented", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_dataset_samples.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 02: System Block Architecture Diagram (Input -> Preprocessing -> Model -> UI)
fig, ax = plt.subplots(figsize=(14, 3.8))
ax.axis("off")

blocks = [
    ("1. Input Source", "File Mode:\nImage (.jpg/.png)\nVideo (.mp4)\n\nReal-time Mode:\nWebcam / CCTV Stream", "#DCEEFB"),
    ("2. Module 1: OpenCV", "Color Transform\n(Gray/HSV/HistEq/Gamma)\nGeometric Transform\n(Rotate/Scale/Flip)\nImage Smoothing\n(Mean/Gauss/Med/Bilat)", "#D5F5E3"),
    ("3. ROI & Preprocess", "Banknote ROI (1.9:1)\nResize (64 / 128 / 224)\nNormalize [0, 1]\nHWC -> CHW Tensor\n(Flatten for MLP)", "#FCF3CF"),
    ("4. Module 2: AI Model", "Multi-Class Classifier\n(C = 6 VND Classes)\n- ImageMLP (Lab 2)\n- CurrencyCNN\n- MobileNetV2 (TL)\nSoftmax + EMA", "#FADBD8"),
    ("5. Interactive UI", "Streamlit Dashboard\nOriginal vs. Processed\nTop-1 + Conf% + Time(ms)\n6-Class Prob Bar Chart\nSnapshot & Video Export", "#E8DAEF"),
]

box_w, box_h, gap = 2.35, 2.5, 0.55
for idx, (title, body, color) in enumerate(blocks):
    x0 = idx * (box_w + gap)
    rect = patches.FancyBboxPatch(
        (x0, 0.4), box_w, box_h,
        boxstyle="round,pad=0.1,rounding_size=0.15",
        linewidth=1.6, edgecolor="#2C3E50", facecolor=color,
    )
    ax.add_patch(rect)
    ax.text(x0 + box_w / 2, 2.55, title, ha="center", va="center", fontsize=9.5, fontweight="bold", color="#1B2631")
    ax.text(x0 + box_w / 2, 1.45, body, ha="center", va="center", fontsize=8.0, color="#212F3C", linespacing=1.35)
    if idx < len(blocks) - 1:
        ax.annotate(
            "",
            xy=(x0 + box_w + gap - 0.12, 1.65),
            xytext=(x0 + box_w + 0.12, 1.65),
            arrowprops={"arrowstyle": "->", "lw": 2.2, "color": "#2C3E50"},
        )

ax.set_xlim(-0.3, len(blocks) * (box_w + gap) - gap + 0.3)
ax.set_ylim(0.1, 3.3)
ax.set_title("End-to-End Multi-Class Vietnamese Currency Image & Video Processing System Architecture", fontsize=12, fontweight="bold", pad=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_system_architecture.png", dpi=220, bbox_inches="tight")
plt.show()

## 3. Helper Plotting Utilities (Loss Curves & 6×6 Confusion Matrix)


In [ ]:
def plot_training_curves(history: TrainingHistory, title: str, save_path: Path) -> None:
    """Plot Train vs. Validation Cross-Entropy Loss and Accuracy (%) curves."""
    epochs = np.arange(1, len(history.train_losses) + 1)
    fig, (ax_l, ax_a) = plt.subplots(1, 2, figsize=(12, 4.4))

    ax_l.plot(epochs, history.train_losses, label="Train Loss", marker="o", markersize=3)
    ax_l.plot(epochs, history.val_losses, label="Val Loss", marker="s", markersize=3)
    if history.best_epoch > 0:
        ax_l.axvline(history.best_epoch, color="gray", linestyle="--", alpha=0.75, label=f"Best Epoch ({history.best_epoch})")
    ax_l.set_title("Cross-Entropy Loss over Epochs", fontweight="bold")
    ax_l.set_xlabel("Epoch")
    ax_l.set_ylabel("Loss")
    ax_l.grid(True, alpha=0.3)
    ax_l.legend()

    ax_a.plot(epochs, history.train_accs, label="Train Acc (%)", marker="o", markersize=3)
    ax_a.plot(epochs, history.val_accs, label="Val Acc (%)", marker="s", markersize=3)
    ax_a.set_title("Accuracy (%) over Epochs", fontweight="bold")
    ax_a.set_xlabel("Epoch")
    ax_a.set_ylabel("Accuracy (%)")
    ax_a.set_ylim(0, 105)
    ax_a.grid(True, alpha=0.3)
    ax_a.legend()

    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_confusion_matrix_6x6(metrics: dict, title: str, save_path: Path) -> None:
    """Plot annotated 6x6 multi-class Confusion Matrix (counts + row recall %)."""
    cm = metrics["cm_array"]
    display_labels = [CLASS_DISPLAY_NAMES.get(c, c) for c in metrics["class_names"]]
    n_cls = len(display_labels)

    fig, ax = plt.subplots(figsize=(8.2, 6.8))
    im = ax.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(n_cls), labels=display_labels, rotation=30, ha="right")
    ax.set_yticks(range(n_cls), labels=display_labels)
    ax.set_xlabel("Predicted Label", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    ax.set_title(
        f"{title}\nAcc: {metrics['accuracy_pct']:.2f}% | Macro F1: {metrics['macro_f1']:.4f}",
        fontsize=12,
        fontweight="bold",
        pad=10,
    )

    thresh = cm.max() / 2.0
    row_sums = cm.sum(axis=1, keepdims=True)
    for i in range(n_cls):
        for j in range(n_cls):
            cnt = cm[i, j]
            row_pct = 100.0 * cnt / max(int(row_sums[i, 0]), 1)
            color = "white" if cnt > thresh else "black"
            ax.text(j, i, f"{cnt}\n({row_pct:.0f}%)", ha="center", va="center", color=color, fontsize=9, fontweight="bold")

    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

## 4. Train Model 1 — `MultiClassImageMLP` (Lab #02 Upgrade on Cropped Banknotes)


In [ ]:
seed_everything(42)
cfg_mlp = Lab3Config(
    data_dir=str(DATA_DIR),
    model_type="mlp",
    img_size=64,
    hidden_dims=(512, 256, 128),
    dropout=0.25,
    batch_size=32,
    num_workers=OPTIMAL_WORKERS,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=45,
    seed=42,
)

tr_mlp, va_mlp, te_mlp, _ = create_dataloaders(cfg_mlp)
mlp_model = build_model(cfg_mlp)
mlp_trainer = MultiClassTrainer(mlp_model, cfg_mlp)

print(f"=== Training MultiClassImageMLP | Params: {mlp_model.count_parameters():,} | Device: {cfg_mlp.device} ===")
t0 = time.perf_counter()
mlp_history = mlp_trainer.fit(tr_mlp, va_mlp, verbose=True)
mlp_train_sec = time.perf_counter() - t0
mlp_metrics = mlp_trainer.test(te_mlp)
print("\n" + mlp_metrics["summary"])

mlp_ckpt_path = mlp_trainer.save_checkpoint(OUTPUT_DIR / "best_mlp.pt", test_metrics=mlp_metrics)
plot_training_curves(mlp_history, "MultiClassImageMLP (64x64, Cropped) — Loss & Accuracy", OUTPUT_DIR / "03_mlp_loss_curves.png")
plot_confusion_matrix_6x6(mlp_metrics, "MultiClassImageMLP — Confusion Matrix", OUTPUT_DIR / "04_mlp_confusion_matrix.png")

## 5. Train Model 2 — `CurrencyCNN` (From-Scratch 2D CNN, 128×128)


In [ ]:
seed_everything(42)
cfg_cnn = Lab3Config(
    data_dir=str(DATA_DIR),
    model_type="cnn",
    img_size=128,
    dropout=0.25,
    batch_size=32,
    num_workers=OPTIMAL_WORKERS,
    lr=3e-4,
    weight_decay=1e-4,
    epochs=50,
    seed=42,
)

tr_cnn, va_cnn, te_cnn, _ = create_dataloaders(cfg_cnn)
cnn_model = build_model(cfg_cnn)
cnn_trainer = MultiClassTrainer(cnn_model, cfg_cnn)

print(f"=== Training CurrencyCNN | Params: {cnn_model.count_parameters():,} | Device: {cfg_cnn.device} ===")
t0 = time.perf_counter()
cnn_history = cnn_trainer.fit(tr_cnn, va_cnn, verbose=True)
cnn_train_sec = time.perf_counter() - t0
cnn_metrics = cnn_trainer.test(te_cnn)
print("\n" + cnn_metrics["summary"])

cnn_ckpt_path = cnn_trainer.save_checkpoint(OUTPUT_DIR / "best_cnn.pt", test_metrics=cnn_metrics)
plot_training_curves(cnn_history, "CurrencyCNN (128x128, Cropped) — Loss & Accuracy", OUTPUT_DIR / "05_cnn_loss_curves.png")
plot_confusion_matrix_6x6(cnn_metrics, "CurrencyCNN — Confusion Matrix", OUTPUT_DIR / "06_cnn_confusion_matrix.png")

## 6. Train Model 3 — `CurrencyMobileNetV2` (Transfer Learning, 224×224 Flagship)


In [ ]:
seed_everything(42)
cfg_mob = Lab3Config(
    data_dir=str(DATA_DIR),
    model_type="mobilenet",
    img_size=224,
    dropout=0.25,
    batch_size=32,
    num_workers=OPTIMAL_WORKERS,
    lr=2e-4,
    weight_decay=1e-4,
    epochs=25,
    seed=42,
)

tr_mob, va_mob, te_mob, _ = create_dataloaders(cfg_mob)
mob_model = build_model(cfg_mob, pretrained=True)
mob_trainer = MultiClassTrainer(mob_model, cfg_mob)

print(f"=== Training CurrencyMobileNetV2 | Params: {mob_model.count_parameters():,} | Device: {cfg_mob.device} ===")
t0 = time.perf_counter()
mob_history = mob_trainer.fit(tr_mob, va_mob, verbose=True)
mob_train_sec = time.perf_counter() - t0
mob_metrics = mob_trainer.test(te_mob)
print("\n" + mob_metrics["summary"])

mob_ckpt_path = mob_trainer.save_checkpoint(OUTPUT_DIR / "best_mobilenet.pt", test_metrics=mob_metrics)
print(f"Saved MobileNetV2 checkpoint: {mob_ckpt_path} ({mob_ckpt_path.stat().st_size / (1024*1024):.2f} MB)")

plot_training_curves(mob_history, "CurrencyMobileNetV2 (224x224, Pretrained) — Loss & Accuracy", OUTPUT_DIR / "05b_mobilenet_loss_curves.png")
plot_confusion_matrix_6x6(mob_metrics, "CurrencyMobileNetV2 (Pretrained) — Confusion Matrix", OUTPUT_DIR / "06b_mobilenet_confusion_matrix.png")

## 7. Comparative Analysis #1 — Binary (Lab #02) vs. Multi-Class (Lab #03) & Architecture Benchmark


In [ ]:
class BinaryBaselineMLP(nn.Module):
    """Exact Lab #02 binary 1-logit MLP head for fair Binary vs. Multi-Class benchmarking."""

    def __init__(self, input_dim: int, hidden_dims: tuple[int, ...] = (512, 256, 128), dropout: float = 0.25) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim > 2:
            x = x.flatten(start_dim=1)
        return self.network(x).squeeze(-1)


def benchmark_binary_vs_multiclass(device_str: str, steps: int = 30) -> dict:
    device = torch.device(device_str)
    seed_everything(42)

    bin_model = BinaryBaselineMLP(input_dim=3 * 64 * 64).to(device)
    mc_model = MultiClassImageMLP(input_dim=3 * 64 * 64, num_classes=6).to(device)
    bin_opt = torch.optim.Adam(bin_model.parameters(), lr=1e-4)
    mc_opt = torch.optim.Adam(mc_model.parameters(), lr=1e-4)
    bce_loss, ce_loss = nn.BCEWithLogitsLoss(), nn.CrossEntropyLoss()

    x_dummy = torch.randn(32, 3 * 64 * 64, device=device)
    y_bin = torch.randint(0, 2, (32,), dtype=torch.float32, device=device)
    y_mc = torch.randint(0, 6, (32,), dtype=torch.long, device=device)

    for _ in range(3):
        bin_opt.zero_grad(); bce_loss(bin_model(x_dummy), y_bin).backward(); bin_opt.step()
        mc_opt.zero_grad(); ce_loss(mc_model(x_dummy), y_mc).backward(); mc_opt.step()

    def measure(model, opt, loss_fn, y_target):
        losses, grad_norms = [], []
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(steps):
            opt.zero_grad()
            loss = loss_fn(model(x_dummy), y_target)
            loss.backward()
            out_layer = [m for m in model.modules() if isinstance(m, nn.Linear)][-1]
            grad_norms.append(float(out_layer.weight.grad.norm().item()))
            losses.append(float(loss.item()))
            opt.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        ms_per_step = (time.perf_counter() - t0) * 1000.0 / steps
        out_layer = [m for m in model.modules() if isinstance(m, nn.Linear)][-1]
        return {
            "total_params": sum(p.numel() for p in model.parameters()),
            "output_head_params": out_layer.weight.numel() + out_layer.bias.numel(),
            "avg_ms_per_step": ms_per_step,
            "initial_loss": losses[0],
            "mean_out_grad_norm": float(np.mean(grad_norms)),
        }

    return {
        "binary_lab02": measure(bin_model, bin_opt, bce_loss, y_bin),
        "multiclass_lab03": measure(mc_model, mc_opt, ce_loss, y_mc),
    }


comp = benchmark_binary_vs_multiclass(get_device())
b_res, m_res = comp["binary_lab02"], comp["multiclass_lab03"]

print("=== 3-Architecture Comparison on Cropped 6-Class VND Test Set ===")
print(f"MultiClassImageMLP : Params={mlp_model.count_parameters():>9,} | Acc={mlp_metrics['accuracy_pct']:.2f}% | Macro F1={mlp_metrics['macro_f1']:.4f} | Train={mlp_train_sec:.1f}s")
print(f"CurrencyCNN        : Params={cnn_model.count_parameters():>9,} | Acc={cnn_metrics['accuracy_pct']:.2f}% | Macro F1={cnn_metrics['macro_f1']:.4f} | Train={cnn_train_sec:.1f}s")
print(f"CurrencyMobileNetV2: Params={mob_model.count_parameters():>9,} | Acc={mob_metrics['accuracy_pct']:.2f}% | Macro F1={mob_metrics['macro_f1']:.4f} | Train={mob_train_sec:.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.3))
axes[0].bar(["Binary (C=2)", "Multi-Class (C=6)"], [b_res["output_head_params"], m_res["output_head_params"]], color=["#5DADE2", "#E74C3C"])
axes[0].set_title("Output Head Parameters", fontweight="bold")
axes[0].set_ylabel("Parameter Count")

axes[1].bar(["Binary (BCE)", "Multi-Class (CE)"], [np.log(2), np.log(6)], color=["#5DADE2", "#E74C3C"])
axes[1].set_title("Theoretical Initial Loss ln(C)", fontweight="bold")
axes[1].set_ylabel("Nats")

axes[2].bar(
    ["MLP\n(6.46M)", "CNN\n(243K)", "MobileNetV2\n(2.23M TL)"],
    [mlp_metrics["accuracy_pct"], cnn_metrics["accuracy_pct"], mob_metrics["accuracy_pct"]],
    color=["#F5B041", "#3498DB", "#2ECC71"],
)
axes[2].set_title("Cropped 6-Class Test Accuracy (%)", fontweight="bold")
axes[2].set_ylim(0, 108)
axes[2].set_ylabel("Accuracy (%)")
for bar in axes[2].patches:
    h = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width() / 2, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=9)

for ax in axes:
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Comparative Analysis: Binary vs. Multi-Class & MLP vs. CNN vs. MobileNetV2", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_binary_vs_multiclass.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Comparative Analysis #2 — Impact of OpenCV Filters (Module 1) on Flagship MobileNetV2 (Module 2)


In [ ]:
engine = CurrencyInferenceEngine(mob_ckpt_path, device=get_device())

filter_scenarios: dict[str, ImageProcessingParams] = {
    "Original (No Filter)": ImageProcessingParams(),
    "Color: Grayscale": ImageProcessingParams(color_mode="grayscale"),
    "Color: Hist Eq (YCrCb)": ImageProcessingParams(color_mode="hist_eq"),
    "Color: Dark (-50)": ImageProcessingParams(brightness=-50),
    "Color: Gamma (1.8)": ImageProcessingParams(gamma=1.8),
    "Geom: Rotate (+25 deg)": ImageProcessingParams(angle=25.0),
    "Smooth: Gaussian (k=7)": ImageProcessingParams(smooth_method="gaussian", kernel_size=7, sigma=1.8),
    "Smooth: Median (k=7)": ImageProcessingParams(smooth_method="median", kernel_size=7),
}

test_rgb_images = []
for p in x_te:
    with Image.open(p) as im:
        test_rgb_images.append(np.array(im.convert("RGB")))

filter_results = {}
print(f"{'Filter Scenario':<26} | {'Acc (%)':>8} | {'Macro F1':>8} | {'Mean Conf%':>10} | {'Avg Time(ms)':>12}")
print("-" * 78)

for scenario_name, f_params in filter_scenarios.items():
    probs_list, latencies = [], []
    for img_rgb in test_rgb_images:
        res = engine.predict_frame(img_rgb, params=f_params)
        p_vec = [res.probabilities[CLASS_DISPLAY_NAMES[c]] / 100.0 for c in DEFAULT_CLASS_NAMES]
        probs_list.append(p_vec)
        latencies.append(res.total_ms)

    m = evaluate_multiclass_metrics(y_te, np.asarray(probs_list), DEFAULT_CLASS_NAMES)
    avg_ms = float(np.mean(latencies))
    filter_results[scenario_name] = {
        "accuracy_pct": m["accuracy_pct"],
        "macro_f1": m["macro_f1"],
        "mean_confidence_pct": m["mean_confidence_pct"],
        "avg_ms": avg_ms,
    }
    print(
        f"{scenario_name:<26} | "
        f"{m['accuracy_pct']:>7.2f}% | "
        f"{m['macro_f1']:>8.4f} | "
        f"{m['mean_confidence_pct']:>9.2f}% | "
        f"{avg_ms:>10.2f} ms"
    )

names = list(filter_results.keys())
accs = [filter_results[k]["accuracy_pct"] for k in names]
confs = [filter_results[k]["mean_confidence_pct"] for k in names]
x_pos = np.arange(len(names))
bar_w = 0.38

fig, ax = plt.subplots(figsize=(12.5, 5.2))
b1 = ax.bar(x_pos - bar_w / 2, accs, width=bar_w, label="Test Accuracy (%)", color="#2E86C1")
b2 = ax.bar(x_pos + bar_w / 2, confs, width=bar_w, label="Mean Top-1 Confidence (%)", color="#F39C12")

ax.set_xticks(x_pos, labels=names, rotation=22, ha="right")
ax.set_ylim(0, 110)
ax.set_ylabel("Percentage (%)")
ax.set_title("Impact of Traditional OpenCV Filters (Module 1) on Multi-Class MobileNetV2 (Module 2)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.legend()

for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 1.2, f"{h:.1f}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_opencv_filter_impact.png", dpi=200, bbox_inches="tight")
plt.show()

summary_payload = {
    "mlp_test": {k: v for k, v in mlp_metrics.items() if k not in {"cm_array", "summary"}},
    "cnn_test": {k: v for k, v in cnn_metrics.items() if k not in {"cm_array", "summary"}},
    "mobilenet_test": {k: v for k, v in mob_metrics.items() if k not in {"cm_array", "summary"}},
    "binary_vs_multiclass": comp,
    "opencv_filter_impact": filter_results,
}
summary_file = OUTPUT_DIR / "experiment_summary.json"
summary_file.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
print(f"Saved full experiment summary JSON to: {summary_file}")